In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

def create_figure_and_df(analysis_df, dimension_ui_name, drilldown_selection, dimension_config, order_map):
    """
    제안 3: 조직 세대교체 현황 (직위별 연령 분포) 그래프 및 피벗 테이블을 생성합니다.
    """
    if analysis_df.empty:
        fig = go.Figure().update_layout(title_text="분석할 데이터가 없습니다.")
        return fig, pd.DataFrame()

    config = dimension_config.get(dimension_ui_name, {})
    position_order = order_map.get('POSITION_NAME', [])

    # 2. 차원 설정에 따라 그룹핑/컬러링에 사용할 컬럼과 데이터 결정
    if config.get('type') == 'hierarchical' and drilldown_selection != '전체':
        top_level_col = config.get('top')
        grouping_col = config.get('sub')
        plot_df = analysis_df[analysis_df[top_level_col] == drilldown_selection]
        title_text = f"'{drilldown_selection}' 내 직위별 연령 분포"
        legend_title = grouping_col
        group_order = order_map.get(grouping_col, sorted(plot_df[grouping_col].unique()))
    else:
        plot_df = analysis_df
        grouping_col = config.get('top', config.get('col'))
        title_text = f"전체 {dimension_ui_name} 직위별 연령 분포"
        legend_title = grouping_col
        group_order = order_map.get(grouping_col, sorted(plot_df[grouping_col].unique()))
        
    # 3. 그래프 생성
    fig = go.Figure()
    colors = px.colors.qualitative.Plotly
    for i, group_name in enumerate(group_order):
        if group_name in plot_df[grouping_col].unique():
            df_filtered = plot_df[plot_df[grouping_col] == group_name]
            fig.add_trace(go.Box(x=df_filtered['POSITION_NAME'], y=df_filtered['AGE'], name=str(group_name), marker_color=colors[i % len(colors)], boxpoints='outliers'))

    # 4. 레이아웃 업데이트
    y_min, y_max = (plot_df['AGE'].min(), plot_df['AGE'].max()) if not plot_df.empty else (20, 60)
    fixed_y_range = [y_min - 5, y_max + 5]
    fig.update_layout(
        template='plotly', title_text=title_text, xaxis_title='직위', yaxis_title='연령',
        font_size=14, height=700, boxmode='group', legend_title_text=legend_title,
        yaxis_range=fixed_y_range, xaxis=dict(categoryorder='array', categoryarray=position_order)
    )
    
    # 5. 요약 테이블(aggregate_df) 생성
    pivot_col = config.get('top', config.get('col'))
    if pivot_col and pivot_col in analysis_df.columns:
        aggregate_df = analysis_df.pivot_table(index='POSITION_NAME', columns=pivot_col, values='AGE', aggfunc='mean', observed=True)
        aggregate_df['전체 평균'] = analysis_df.groupby('POSITION_NAME', observed=True)['AGE'].mean()
        pivot_order = order_map.get(pivot_col, [])
        cols = ['전체 평균'] + [col for col in pivot_order if col in aggregate_df.columns]
        aggregate_df = aggregate_df[cols]
        aggregate_df = aggregate_df.reindex(position_order).round(2).fillna('-')
    else:
        aggregate_df = pd.DataFrame()

    return fig, aggregate_df